# English audio render — Don's voice (F5), commit-per-article

Re-renders the **English** narration for every audio-eligible library + blog article in *your*
cloned voice, and **commits + pushes each article's MP3 the moment it finishes** — so if Colab
disconnects you never lose more than the one article in flight. To resume: just re-run cells 1–6,
then the render cell again; it skips everything already rendered.

Run cells **top to bottom** (Shift+Enter). First: `Runtime → Change runtime type → T4 GPU → Save`.

Your voice reference is already in the repo (`scripts/voice-refs/don-reference.m4a`), so there's
nothing to upload. Total time depends on article count — plan for a few hours on a T4.

## 1. Confirm GPU
Output should mention `Tesla T4` (or similar).

In [ ]:
!nvidia-smi

## 2. Install Node 20 + F5-TTS + ffmpeg (~2 min)
`render-post-audio.mjs` needs modern Node + ffmpeg; F5-TTS does the voice cloning (jieba is an
undeclared F5 dep on fresh Colab images).

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs ffmpeg -qq > /dev/null
!node --version && ffmpeg -version | head -1
!pip install -q f5-tts jieba

## 3. Clone the branch with your token + configure git
Paste a GitHub token with **repo** scope when prompted (it is read with `getpass`, so it is
**not** saved in the notebook). The token stays in this throwaway Colab's git config only, so the
per-article `git push` can authenticate.

In [ ]:
import os, getpass, subprocess
BRANCH = 'claude/compassionate-dirac-rdkw22'
REPO   = 'github.com/Donwonmagic/potentially-profitable.git'
token  = getpass.getpass('GitHub token (repo scope, not stored): ').strip()
url    = f'https://x-access-token:{token}@{REPO}'
!rm -rf /content/potentially-profitable
subprocess.run(['git','clone','-b',BRANCH,'--depth','1',url,'/content/potentially-profitable'], check=True)
%cd /content/potentially-profitable
!git config user.name 'Don Goldstein'
!git config user.email 'dongoldstein.accts@gmail.com'
# F5 hardcodes torch.xpu (Intel GPU); Colab's CUDA build doesn't ship it. Make the check safe.
import glob
for f in glob.glob('/usr/local/lib/python3*/dist-packages/f5_tts/**/*.py', recursive=True):
    s = open(f, encoding='utf-8').read()
    if 'torch.xpu.is_available()' in s:
        open(f,'w',encoding='utf-8').write(s.replace('torch.xpu.is_available()', '(hasattr(torch,"xpu") and torch.xpu.is_available())'))
print('cloned', BRANCH, '+ F5 patched')

## 4. Verify your voice reference is in the repo

In [ ]:
import os
EN_REF = 'scripts/voice-refs/don-reference.m4a'
EN_TXT = 'scripts/voice-refs/don-reference.txt'
print('voice clip   :', os.path.isfile(EN_REF), '(', os.path.getsize(EN_REF) if os.path.isfile(EN_REF) else 0, 'bytes )')
print('transcript   :', os.path.isfile(EN_TXT))
assert os.path.isfile(EN_REF) and os.path.isfile(EN_TXT), 'voice reference missing — something is wrong with the clone'

## 5. Build the article list
Every English library + blog article with an audio (listen) button. To render only a subset,
edit `targets` (e.g. `targets = ['library/restaurant-menu-engineering', 'blog/restaurant-menu-inflation-2026']`).

`CUTOFF` defines "already done": any article whose `audio.json` was generated by F5 on/after this
timestamp is skipped on a re-run — that's what makes the render resumable after a disconnect.

In [ ]:
import glob, os
def has_listen(p):
    try: return 'id="listen-btn"' in open(p, encoding='utf-8').read()
    except Exception: return False
targets = []
for base in ('library','blog'):
    for d in sorted(glob.glob(base + '/*/')):
        idx = os.path.join(d, 'index.html')
        if os.path.isfile(idx) and has_listen(idx):
            targets.append(d.rstrip('/'))
CUTOFF = '2026-06-22T00:00:00'   # treat F5 renders on/after this as already done
print(len(targets), 'audio-eligible English articles')
for t in targets: print('  ', t)

## 6. Render — the long one (hours)
For each article: if it's already F5-rendered on/after `CUTOFF`, skip; otherwise render the English
track in your cloned voice and **commit + push it immediately**. A failure on one article is logged
and the loop continues. The F5 base model (~1.5 GB) downloads on the first article only.

**If Colab disconnects:** re-run cells 1–6. The re-clone already contains every article you pushed,
so this cell resumes from where it stopped.

In [ ]:
import os, json, subprocess
def is_done(d):
    j = os.path.join(d, 'audio.json')
    if not os.path.isfile(j): return False
    try: m = json.load(open(j, encoding='utf-8'))
    except Exception: return False
    return str(m.get('engine','')) == 'f5' and str(m.get('generatedAt','')) >= CUTOFF

done = skipped = failed = 0
fails = []
for i, d in enumerate(targets, 1):
    if is_done(d):
        skipped += 1; print(f'[{i}/{len(targets)}] skip (already f5): {d}'); continue
    print(f'[{i}/{len(targets)}] render + commit: {d}', flush=True)
    r = subprocess.run(['node','scripts/render-post-audio.mjs', d,
                        '--engine','f5','--languages','en',
                        '--force-retranslate','--commit-per-article'])
    if r.returncode == 0: done += 1
    else: failed += 1; fails.append(d); print(f'  !! failed (exit {r.returncode}) — continuing')
print(f'\n=== rendered {done}, skipped {skipped}, failed {failed} ===')
if fails: print('failed:', fails)

## 7. Done
Every rendered article is already committed and pushed to `claude/compassionate-dirac-rdkw22`
(commit subject `audio: <slug>`). Nothing to download, nothing to commit by hand — the live player
picks up the new MP3s on the next deploy.

To also render Spanish later: the same script handles `--languages es` (it renders the native
Spanish under `/es/` and clones your `don-reference.es.m4a`; Spanish needs the F5-Spanish checkpoint
— see `scripts/voice-refs/README.md`). English first, as requested.